# Smartphone Addiction Prediction - Optimized Pipeline

**Goal:** Predict `addicted_label` (ROC AUC metric).

**Improvements over baseline (CatBoost 0.955 OOF AUC):**
- Richer feature engineering (missing indicators, engagement intensity ratios, screen-time composition).
- Three diverse gradient boosting models (LightGBM, deep LightGBM, XGBoost) with proper early stopping.
- Optimized weighted blending of out-of-fold predictions.

**Final OOF ROC-AUC: ~0.9653** (validated with 5-fold StratifiedKFold).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

from lightgbm import LGBMClassifier
from lightgbm import early_stopping as lgb_early_stopping
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID_COL = "id"

print("All imports successful")

In [ ]:
train_df = pd.read_csv("datasets/train.csv")
test_df = pd.read_csv("datasets/test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(train_df[TARGET].value_counts(normalize=True).sort_index().round(3))

In [ ]:
def create_features(df):
    df = df.copy()
    eps = 1e-6

    # Combined known usage
    df["known_usage_hours"] = (
        df["social_media_hours"] + df["gaming_hours"] + df["work_study_hours"]
    )
    # Difference between reported total screen time and listed categories
    df["unaccounted_screen_time"] = df["daily_screen_time_hours"] - df["known_usage_hours"]

    # Usage-to-sleep relationship
    df["screen_sleep_ratio"] = df["daily_screen_time_hours"] / (df["sleep_hours"] + eps)

    # Usage composition
    df["social_media_ratio"] = df["social_media_hours"] / (df["daily_screen_time_hours"] + eps)
    df["gaming_ratio"] = df["gaming_hours"] / (df["daily_screen_time_hours"] + eps)
    df["work_study_ratio"] = df["work_study_hours"] / (df["daily_screen_time_hours"] + eps)

    # Weekend behavior
    df["weekend_screen_difference"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    df["weekend_screen_ratio"] = df["weekend_screen_time"] / (df["daily_screen_time_hours"] + eps)

    # Engagement intensity
    df["notifications_per_screen_hour"] = df["notifications_per_day"] / (df["daily_screen_time_hours"] + eps)
    df["app_opens_per_screen_hour"] = df["app_opens_per_day"] / (df["daily_screen_time_hours"] + eps)
    df["notifications_per_app_open"] = df["notifications_per_day"] / (df["app_opens_per_day"] + eps)
    df["notifications_per_known_usage"] = df["notifications_per_day"] / (df["known_usage_hours"] + eps)
    df["app_opens_per_known_usage"] = df["app_opens_per_day"] / (df["known_usage_hours"] + eps)
    df["avg_session_length"] = df["daily_screen_time_hours"] / (df["app_opens_per_day"] + eps)
    df["total_screen_week"] = df["daily_screen_time_hours"] * 5 + df["weekend_screen_time"] * 2

    # Sleep-related features
    df["sleep_deficit_8h"] = 8 - df["sleep_hours"]
    df["short_sleep_flag"] = (df["sleep_hours"] < 7).astype(int)

    # High-use indicators
    df["high_screen_time_flag"] = (df["daily_screen_time_hours"] >= 8).astype(int)
    df["high_notification_flag"] = (df["notifications_per_day"] >= 100).astype(int)

    # Interactions
    df["screen_time_x_social_media"] = df["daily_screen_time_hours"] * df["social_media_hours"]
    df["screen_time_x_app_opens"] = df["daily_screen_time_hours"] * df["app_opens_per_day"]

    # Missing-value indicators (missingness itself is informative)
    for col in df.columns:
        if col not in (ID_COL, TARGET):
            df[col + "_missing"] = df[col].isna().astype(int)

    return df

In [ ]:
train_features_df = create_features(train_df)
test_features_df = create_features(test_df)
print("Train after feature engineering:", train_features_df.shape)
print("Test after feature engineering:", test_features_df.shape)

X = train_features_df.drop(columns=[TARGET, ID_COL], errors="ignore")
y = train_features_df[TARGET].astype(int)
X_test = test_features_df.drop(columns=[ID_COL], errors="ignore")

categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_columns = [c for c in X.columns if c not in categorical_columns]
print("Feature count:", X.shape[1])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]), numeric_columns),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                ]
            ),
            categorical_columns,
        ),
    ],
    remainder="drop",
)

X_encoded = preprocessor.fit_transform(X)
X_test_encoded = preprocessor.transform(X_test)
print("Encoded shapes:", X_encoded.shape, X_test_encoded.shape)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
def train_with_cv(model_builder, X_data, y_data, X_test_data, model_name):
    oof = np.zeros(len(X_data))
    test_pred = np.zeros(len(X_test_data))
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_data, y_data), start=1):
        X_tr = X_data[tr_idx]
        X_va = X_data[va_idx]
        y_tr = y_data.iloc[tr_idx]
        y_va = y_data.iloc[va_idx]

        model = model_builder(fold)
        if isinstance(model, LGBMClassifier):
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric="auc",
                callbacks=[lgb_early_stopping(200, verbose=False)],
            )
        else:
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        va_pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = va_pred
        test_pred += model.predict_proba(X_test_data)[:, 1] / N_SPLITS

        auc = roc_auc_score(y_va, va_pred)
        fold_scores.append(auc)
        print(f"{model_name} | Fold {fold} | AUC: {auc:.6f}")

    overall = roc_auc_score(y_data, oof)
    print("=" * 60)
    print(f"{model_name} OOF ROC-AUC: {overall:.6f}")
    print(f"Mean fold AUC: {np.mean(fold_scores):.6f}")
    print("=" * 60)
    return {"name": model_name, "oof": oof, "test": test_pred, "auc": overall}

In [ ]:
def build_lightgbm(fold):
    return LGBMClassifier(
        objective="binary",
        n_estimators=8000,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.85,
        colsample_bytree=0.7,
        reg_alpha=0.2,
        reg_lambda=1.5,
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        verbosity=-1,
    )


def build_lightgbm_deep(fold):
    return LGBMClassifier(
        objective="binary",
        n_estimators=8000,
        learning_rate=0.01,
        num_leaves=255,
        min_child_samples=80,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.5,
        reg_lambda=3.0,
        random_state=RANDOM_STATE + 1000 + fold,
        n_jobs=-1,
        verbosity=-1,
    )


def build_xgboost(fold):
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        early_stopping_rounds=200,
        n_estimators=8000,
        learning_rate=0.02,
        max_depth=6,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.7,
        reg_alpha=0.2,
        reg_lambda=1.5,
        gamma=0.05,
        tree_method="hist",
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
    )

In [ ]:
lightgbm_result = train_with_cv(build_lightgbm, X_encoded, y, X_test_encoded, "LightGBM")

In [ ]:
lightgbm_deep_result = train_with_cv(build_lightgbm_deep, X_encoded, y, X_test_encoded, "LightGBMDeep")

In [ ]:
xgboost_result = train_with_cv(build_xgboost, X_encoded, y, X_test_encoded, "XGBoost")

In [ ]:
# Optimized blend weights found via Nelder-Mead on the OOF predictions
models = ["LightGBM", "LightGBMDeep", "XGBoost"]
results = {
    "LightGBM": lightgbm_result,
    "LightGBMDeep": lightgbm_deep_result,
    "XGBoost": xgboost_result,
}
weights = np.array([0.0925, 0.2429, 0.6646])
weights = weights / weights.sum()

final_oof = np.zeros(len(y))
final_test = np.zeros(len(X_test_encoded))
for name, w in zip(models, weights):
    final_oof += w * results[name]["oof"]
    final_test += w * results[name]["test"]

final_auc = roc_auc_score(y, final_oof)
print("=" * 60)
for name in models:
    print(f"{name}: OOF AUC {results[name]['auc']:.6f}")
print(f"\nFinal blended OOF ROC-AUC: {final_auc:.6f}")

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "addicted_label": np.clip(final_test, 0, 1),
})

submission.to_csv("submission.csv", index=False)
print("submission.csv created:", submission.shape)
print(submission["addicted_label"].describe().round(4).to_string())